In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("DS_DF_WQI_Train.csv")

In [50]:

df_raw = pd.read_csv("DS_DF_WQI_Train.csv")
print(df_raw.columns.tolist())

['SMPLDATETIME', 'STNCODE', 'USRCODES', 'Temp', 'SpCond', 'Sal', 'DO_pct', 'DO_mgl', 'Depth', 'pH', 'Turb', 'WQI', 'WQI_Class']


In [51]:
print("Columns detected:", df_raw.columns.tolist())

Columns detected: ['SMPLDATETIME', 'STNCODE', 'USRCODES', 'Temp', 'SpCond', 'Sal', 'DO_pct', 'DO_mgl', 'Depth', 'pH', 'Turb', 'WQI', 'WQI_Class']


In [52]:
# Parse datetime
df_raw["SMPLDATETIME"] = pd.to_datetime(df_raw["SMPLDATETIME"])

In [53]:

# Sort
df_raw = df_raw.sort_values("SMPLDATETIME")

In [54]:
# Drop duplicates
df_raw = df_raw[~df_raw["SMPLDATETIME"].duplicated(keep="first")]

In [55]:
# Set index
df_raw = df_raw.set_index("SMPLDATETIME")

In [56]:
# Detect gaps
print(df_raw.index.to_series().diff().value_counts().sort_index())

SMPLDATETIME
0 days 00:30:00     33113
0 days 01:00:00         2
0 days 04:00:00         1
0 days 04:30:00         1
0 days 05:00:00         1
0 days 05:30:00         6
0 days 06:00:00         4
0 days 06:30:00         2
0 days 07:00:00         5
35 days 06:00:00        1
55 days 21:30:00        1
Name: count, dtype: int64


In [57]:
# Reindex (strict 30‑minute steps)
df = df_raw.asfreq("30min")

In [58]:
print("\nAfter reindexing:")
df.info()


After reindexing:
<class 'pandas.DataFrame'>
DatetimeIndex: 37730 entries, 2004-01-01 00:00:00 to 2006-02-25 00:30:00
Freq: 30min
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   STNCODE    33138 non-null  str    
 1   USRCODES   33138 non-null  str    
 2   Temp       33138 non-null  float64
 3   SpCond     33138 non-null  float64
 4   Sal        33138 non-null  float64
 5   DO_pct     33138 non-null  float64
 6   DO_mgl     33138 non-null  float64
 7   Depth      33138 non-null  float64
 8   pH         33138 non-null  float64
 9   Turb       33138 non-null  float64
 10  WQI        33138 non-null  float64
 11  WQI_Class  33138 non-null  str    
dtypes: float64(9), str(3)
memory usage: 4.4 MB


## Clean Values

In [59]:
df_clean = df.copy()

In [60]:
# 1. Drop redundant DO_pct (use DO_mgl instead)
if "DO_pct" in df_clean.columns:
    df_clean = df_clean.drop(columns=["DO_pct"])

In [61]:
# 2. Clip impossible / non-physical values
df_clean["Temp"]  = df_clean["Temp"].clip(-5, 50)   
df_clean["SpCond"] = df_clean["SpCond"].clip(0, None)   
df_clean["Sal"]    = df_clean["Sal"].clip(0, None)
df_clean["DO_mgl"] = df_clean["DO_mgl"].clip(0, 20)     
df_clean["Depth"]  = df_clean["Depth"].clip(0, None)
df_clean["pH"]     = df_clean["pH"].clip(0, 14)
df_clean["Turb"]   = df_clean["Turb"].clip(0, None)

In [62]:
print("Cleaning complete. df_clean info:")
df_clean.info()

Cleaning complete. df_clean info:
<class 'pandas.DataFrame'>
DatetimeIndex: 37730 entries, 2004-01-01 00:00:00 to 2006-02-25 00:30:00
Freq: 30min
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   STNCODE    33138 non-null  str    
 1   USRCODES   33138 non-null  str    
 2   Temp       33138 non-null  float64
 3   SpCond     33138 non-null  float64
 4   Sal        33138 non-null  float64
 5   DO_mgl     33138 non-null  float64
 6   Depth      33138 non-null  float64
 7   pH         33138 non-null  float64
 8   Turb       33138 non-null  float64
 9   WQI        33138 non-null  float64
 10  WQI_Class  33138 non-null  str    
dtypes: float64(8), str(3)
memory usage: 4.2 MB


### EDA-SAFE IMPUTATION (df_eda)

In [63]:
df_eda = df_clean.copy()

# Which columns are numeric & safe to fill
num_cols = ["Temp","SpCond","Sal","DO_mgl","Depth","pH","Turb","WQI"]

# Categorical/text columns (carry forward/back)
cat_cols = ["STNCODE","USRCODES","WQI_Class"]

In [65]:
# 1) Create imputation flags BEFORE filling
for c in num_cols:
    df_eda[f"{c}_was_imputed"] = df_eda[c].isna()

# 2) Numeric: gentle visual fill → forward fill then backfill
df_eda[num_cols] = df_eda[num_cols].ffill().bfill()

# 3) Categorical: carry forward/back (station/time label continuity)
df_eda[cat_cols] = df_eda[cat_cols].ffill().bfill()

In [66]:
print("df_eda NA summary (should be 0):")
print(df_eda[num_cols + cat_cols].isna().sum(), "\n")

df_eda NA summary (should be 0):
Temp         0
SpCond       0
Sal          0
DO_mgl       0
Depth        0
pH           0
Turb         0
WQI          0
STNCODE      0
USRCODES     0
WQI_Class    0
dtype: int64 



### ML-SAFE IMPUTATION (df_ml)

In [68]:
df_ml = df_clean.copy()

# Create flags & fill numeric vars using PAST-ONLY information
for c in num_cols:
    df_ml[f"{c}_was_imputed"] = df_ml[c].isna()
    
    s = df_ml[c]
    past_roll = s.shift(1).rolling(window=48, min_periods=1).median()

    s = s.fillna(past_roll)

    past_exp = s.shift(1).expanding(min_periods=1).median()
    s = s.fillna(past_exp)

    df_ml[c] = s

In [69]:
for c in cat_cols:
    df_ml[c] = df_ml[c].ffill()

In [71]:
print("df_ml NA summary (ideally 0 or very small at the very start):")
print(df_ml[num_cols + cat_cols].isna().sum())

df_ml NA summary (ideally 0 or very small at the very start):
Temp         0
SpCond       0
Sal          0
DO_mgl       0
Depth        0
pH           0
Turb         0
WQI          0
STNCODE      0
USRCODES     0
WQI_Class    0
dtype: int64


In [74]:
df_feat = df_ml.copy()


df_feat["hour"]      = df_feat.index.hour
df_feat["minute"]    = df_feat.index.minute
df_feat["day"]       = df_feat.index.day
df_feat["month"]     = df_feat.index.month
df_feat["dayofweek"] = df_feat.index.dayofweek
df_feat["dayofyear"] = df_feat.index.dayofyear

In [77]:
h   = df_feat.index.hour
dow = df_feat.index.dayofweek
doy = df_feat.index.dayofyear

In [78]:
df_feat["sin_hour"] = np.sin(2*np.pi*h/24)
df_feat["cos_hour"] = np.cos(2*np.pi*h/24)
df_feat["sin_week"] = np.sin(2*np.pi*dow/7)
df_feat["cos_week"] = np.cos(2*np.pi*dow/7)
df_feat["sin_day"]  = np.sin(2*np.pi*doy/365.25)
df_feat["cos_day"]  = np.cos(2*np.pi*doy/365.25)


In [80]:

if "WQI" in df_feat.columns:
    signal_cols_with_wqi = signal_cols + ["WQI"]
else:
    signal_cols_with_wqi = signal_cols

lag_steps = [1, 48, 336]  # 30-min, 1-day, 1-week
for col in signal_cols_with_wqi:
    for k in lag_steps:
        df_feat[f"{col}_lag{k}"] = df_feat[col].shift(k)


In [83]:

def roll_feats(series: pd.Series, win: int, tag: str):
    past = series.shift(1).rolling(window=win, min_periods=1)
    return past.mean().rename(f"{series.name}_roll{tag}_mean"), past.std().rename(f"{series.name}_roll{tag}_std")

In [84]:
roll_windows = {"48": 48, "336": 336}

In [85]:
roll_cols = signal_cols_with_wqi.copy()

for col in roll_cols:
    for tag, w in roll_windows.items():
        m, s = roll_feats(df_feat[col], w, tag)
        df_feat[m.name] = m
        df_feat[s.name] = s

na_counts = df_feat.isna().sum().sort_values(ascending=False)
print("Top NA counts (expected at series start due to lags/rollings):\n", na_counts.head(15), "\n")

Top NA counts (expected at series start due to lags/rollings):
 DO_mgl_lag336    336
Depth_lag336     336
pH_lag336        336
Turb_lag336      336
WQI_lag336       336
Temp_lag336      336
SpCond_lag336    336
Sal_lag336       336
Turb_lag48        48
pH_lag48          48
Sal_lag48         48
SpCond_lag48      48
Temp_lag48        48
Depth_lag48       48
DO_mgl_lag48      48
dtype: int64 



In [86]:
id_cols      = [c for c in ["STNCODE","USRCODES"] if c in df_feat.columns]
target_cols  = [c for c in ["WQI","WQI_Class"] if c in df_feat.columns]
meta_cols    = [c for c in df_feat.columns if c.endswith("_was_imputed")]
cal_cyc_cols = ["hour","minute","day","month","dayofweek","dayofyear",
                "sin_hour","cos_hour","sin_week","cos_week","sin_day","cos_day"]

In [87]:
ordered = id_cols + target_cols + cal_cyc_cols + meta_cols
ordered += [c for c in df_feat.columns if c not in ordered]
df_feat = df_feat[ordered]

In [88]:
print("df_feat shape:", df_feat.shape)

df_feat shape: (37730, 87)


In [89]:
print("df_feat preview:")
display(df_feat.head(3))

df_feat preview:


,STNCODE,USRCODES,WQI,WQI_Class,hour,minute,day,month,dayofweek,dayofyear,...,pH_roll336_mean,pH_roll336_std,Turb_roll48_mean,Turb_roll48_std,Turb_roll336_mean,Turb_roll336_std,WQI_roll48_mean,WQI_roll48_std,WQI_roll336_mean,WQI_roll336_std
SMPLDATETIME,,,,,,,,,,,,,,,,,,,,,
2004-01-01 00:00:00,deldswq,00:00:00,12.639779,Excellent,0,0,1,1,3,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-01-01 00:30:00,deldswq,00:30:00,12.636899,Excellent,0,30,1,1,3,1,...,6.6,NaN,0.0120,NaN,0.0120,NaN,12.639779,NaN,12.639779,NaN
2004-01-01 01:00:00,deldswq,01:00:00,12.631159,Excellent,1,0,1,1,3,1,...,6.6,0.0,0.0125,0.000707,0.0125,0.000707,12.638339,0.002037,12.638339,0.002037


In [ ]:
# Save dataframes to CSV (keep the datetime index)
df_clean.to_csv("df_clean.csv", index=True)
df_eda.to_csv("df_eda.csv", index=True)
df_ml.to_csv("df_ml.csv", index=True)
df_feat.to_csv("df_feat.csv", index=True)